# 1. Data Exploration - Enron Email Dataset

**Project:** INTELIPS - Intelligent Email Priority System

**Author:** Karim Semaan

**Date:** November 20, 2024

---

## Objectives
1. Load and explore the Enron email dataset
2. Understand data structure and quality
3. Analyze temporal patterns
4. Identify key features for prioritization
5. Prepare data for annotation and modeling

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# Custom utilities
import utils

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1.1 Load Dataset

The Enron email dataset contains approximately 500,000 emails from 150+ corporate users. Let's start by loading a sample to understand the structure.

In [ ]:
# Load a sample first to understand structure (10,000 rows)
DATA_PATH = 'Enron_Dataset/emails.csv'
SAMPLE_SIZE = 10000

print(f"Loading {SAMPLE_SIZE:,} emails for initial exploration...")
df_sample = utils.load_enron_data(DATA_PATH, nrows=SAMPLE_SIZE)

print(f"\nDataset shape: {df_sample.shape}")
print(f"Columns: {list(df_sample.columns)}")

In [ ]:
# Display first few rows
print("First 5 emails:")
df_sample.head()

In [ ]:
# Basic info
print("Dataset Info:")
df_sample.info()

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df_sample.isnull().sum()
missing_pct = (missing / len(df_sample)) * 100
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False))

## 1.2 Exploratory Data Analysis

### 1.2.1 Text Statistics

In [ ]:
# Analyze text lengths
df_sample['subject_length'] = df_sample['subject'].fillna('').str.len()
df_sample['message_length'] = df_sample['message'].fillna('').str.len()

print("Text Length Statistics:")
print("\nSubject Line:")
print(df_sample['subject_length'].describe())
print("\nMessage Body:")
print(df_sample['message_length'].describe())

In [ ]:
# Visualize text length distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subject length
axes[0].hist(df_sample['subject_length'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Subject Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Subject Line Lengths')
axes[0].axvline(df_sample['subject_length'].median(), color='red', linestyle='--', label='Median')
axes[0].legend()

# Message length (log scale for better visualization)
axes[1].hist(np.log10(df_sample['message_length'] + 1), bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Message Length (log10 characters)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Message Body Lengths (log scale)')
axes[1].axvline(np.log10(df_sample['message_length'].median() + 1), color='red', linestyle='--', label='Median')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nMedian subject length: {df_sample['subject_length'].median():.0f} characters")
print(f"Median message length: {df_sample['message_length'].median():.0f} characters")

### 1.2.2 Temporal Analysis

In [ ]:
# Check if date column exists and convert to datetime
if 'date' in df_sample.columns:
    df_sample['datetime'] = pd.to_datetime(df_sample['date'], errors='coerce')
    
    # Extract temporal features
    temporal_features = utils.extract_temporal_features(df_sample['datetime'])
    df_sample = pd.concat([df_sample, temporal_features], axis=1)
    
    print("Temporal Features Extracted:")
    print(temporal_features.head())
else:
    print("Warning: No 'date' column found in dataset")
    print(f"Available columns: {list(df_sample.columns)}")

In [ ]:
# Temporal distribution analysis
if 'hour' in df_sample.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Emails by hour of day
    hour_counts = df_sample['hour'].value_counts().sort_index()
    axes[0, 0].bar(hour_counts.index, hour_counts.values, edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Hour of Day')
    axes[0, 0].set_ylabel('Number of Emails')
    axes[0, 0].set_title('Email Distribution by Hour of Day')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Emails by day of week
    day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_counts = df_sample['day_of_week'].value_counts().sort_index()
    axes[0, 1].bar(range(7), [day_counts.get(i, 0) for i in range(7)], 
                   tick_label=day_names, edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Day of Week')
    axes[0, 1].set_ylabel('Number of Emails')
    axes[0, 1].set_title('Email Distribution by Day of Week')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # Time segment distribution
    time_segment_counts = df_sample['time_segment'].value_counts()
    axes[1, 0].bar(time_segment_counts.index, time_segment_counts.values, edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Time Segment')
    axes[1, 0].set_ylabel('Number of Emails')
    axes[1, 0].set_title('Email Distribution by Time Segment')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Emails over time (by month)
    if 'year' in df_sample.columns and 'month' in df_sample.columns:
        df_sample['year_month'] = df_sample['datetime'].dt.to_period('M')
        monthly_counts = df_sample['year_month'].value_counts().sort_index()
        axes[1, 1].plot(range(len(monthly_counts)), monthly_counts.values, marker='o')
        axes[1, 1].set_xlabel('Time Period')
        axes[1, 1].set_ylabel('Number of Emails')
        axes[1, 1].set_title('Email Volume Over Time')
        axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print insights
    print("\nTemporal Insights:")
    print(f"Peak hour: {hour_counts.idxmax()}:00 ({hour_counts.max():,} emails)")
    print(f"Quietest hour: {hour_counts.idxmin()}:00 ({hour_counts.min():,} emails)")
    print(f"\nWeekday vs Weekend: {df_sample['is_weekend'].value_counts().to_dict()}")
    print(f"Most common time segment: {df_sample['time_segment'].mode()[0]}")

### 1.2.3 Sender and Recipient Analysis

In [ ]:
# Analyze sender patterns
if 'from' in df_sample.columns:
    sender_counts = df_sample['from'].value_counts()
    
    print(f"Total unique senders: {df_sample['from'].nunique():,}")
    print(f"\nTop 10 senders:")
    print(sender_counts.head(10))
    
    # Visualize top senders
    plt.figure(figsize=(12, 6))
    top_senders = sender_counts.head(15)
    plt.barh(range(len(top_senders)), top_senders.values, edgecolor='black', alpha=0.7)
    plt.yticks(range(len(top_senders)), [s[:30] + '...' if len(s) > 30 else s for s in top_senders.index])
    plt.xlabel('Number of Emails Sent')
    plt.ylabel('Sender')
    plt.title('Top 15 Email Senders')
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze recipient patterns
if 'to' in df_sample.columns:
    # Count recipients per email
    df_sample['recipient_count'] = df_sample['to'].fillna('').str.count('@')
    
    print("Recipient Count Statistics:")
    print(df_sample['recipient_count'].describe())
    
    # Visualize recipient distribution
    plt.figure(figsize=(10, 6))
    plt.hist(df_sample['recipient_count'], bins=30, edgecolor='black', alpha=0.7)
    plt.xlabel('Number of Recipients')
    plt.ylabel('Frequency')
    plt.title('Distribution of Recipient Counts per Email')
    plt.axvline(df_sample['recipient_count'].median(), color='red', linestyle='--', label='Median')
    plt.legend()
    plt.show()
    
    print(f"\nMedian recipients per email: {df_sample['recipient_count'].median():.0f}")
    print(f"Max recipients in single email: {df_sample['recipient_count'].max():.0f}")

### 1.2.4 Content Analysis - Urgency Keywords

In [ ]:
# Extract metadata features including urgency indicators
metadata_features = utils.extract_metadata_features(df_sample)
df_sample = pd.concat([df_sample, metadata_features], axis=1)

print("Metadata Features Extracted:")
print(metadata_features.head())

# Analyze urgency keyword presence
urgency_pct = (df_sample['has_urgent_keyword'].sum() / len(df_sample)) * 100
print(f"\nEmails with urgency keywords: {df_sample['has_urgent_keyword'].sum():,} ({urgency_pct:.2f}%)")

In [ ]:
# Analyze deadline features
deadline_features_list = []

print("Extracting deadline features from subjects (this may take a moment)...")
for idx, row in df_sample.head(1000).iterrows():  # Process first 1000 for efficiency
    subject_text = row.get('subject', '')
    deadline_feats = utils.extract_deadline_features(subject_text)
    deadline_features_list.append(deadline_feats)

deadline_df = pd.DataFrame(deadline_features_list)

print("\nDeadline Feature Statistics (first 1000 emails):")
print(deadline_df.sum())
print("\nPercentages:")
print((deadline_df.sum() / len(deadline_df)) * 100)

### 1.2.5 Sample Email Review

In [ ]:
# Display sample emails for manual review
def display_email(idx):
    """Display email details in readable format."""
    email = df_sample.iloc[idx]
    print("=" * 80)
    print(f"Email #{idx}")
    print("=" * 80)
    print(f"From: {email.get('from', 'N/A')}")
    print(f"To: {email.get('to', 'N/A')}")
    print(f"Date: {email.get('date', 'N/A')}")
    print(f"Subject: {email.get('subject', 'N/A')}")
    print("\nMessage:")
    print(email.get('message', 'N/A')[:500] + '...' if len(str(email.get('message', ''))) > 500 else email.get('message', 'N/A'))
    print("\n" + "=" * 80)

# Display 3 random emails
print("Sample Emails:\n")
for idx in np.random.choice(len(df_sample), 3, replace=False):
    display_email(idx)
    print()

## 1.3 Feature Engineering Summary

Based on our exploration, we can extract the following features for prioritization:

In [ ]:
# Summarize available features
print("Available Features for Modeling:\n")

feature_categories = {
    'Temporal Features': [
        'hour', 'day_of_week', 'month', 'time_segment', 'is_weekend'
    ],
    'Metadata Features': [
        'subject_length', 'message_length', 'subject_word_count', 
        'message_word_count', 'recipient_count'
    ],
    'Content Features': [
        'has_urgent_keyword', 'has_deadline', 'has_time_constraint'
    ],
    'Text Data': [
        'subject', 'message'
    ],
    'Sender/Recipient': [
        'from', 'to'
    ]
}

for category, features in feature_categories.items():
    print(f"\n{category}:")
    for feat in features:
        if feat in df_sample.columns:
            print(f"  ✓ {feat}")
        else:
            print(f"  ✗ {feat} (not available)")

## 1.4 Data Quality Assessment

In [ ]:
# Assess data quality
print("Data Quality Report:\n")

# Check for completely empty emails
empty_subject = df_sample['subject'].isna() | (df_sample['subject'].str.strip() == '')
empty_message = df_sample['message'].isna() | (df_sample['message'].str.strip() == '')
both_empty = empty_subject & empty_message

print(f"Emails with empty subject: {empty_subject.sum():,} ({(empty_subject.sum()/len(df_sample)*100):.2f}%)")
print(f"Emails with empty message: {empty_message.sum():,} ({(empty_message.sum()/len(df_sample)*100):.2f}%)")
print(f"Emails with both empty: {both_empty.sum():,} ({(both_empty.sum()/len(df_sample)*100):.2f}%)")

# Check for duplicate emails
if 'message_length' in df_sample.columns:
    potential_duplicates = df_sample[df_sample.duplicated(subset=['from', 'subject', 'message_length'], keep=False)]
    print(f"\nPotential duplicate emails: {len(potential_duplicates):,} ({(len(potential_duplicates)/len(df_sample)*100):.2f}%)")

print("\nRecommendation:")
if both_empty.sum() > 0:
    print(f"  - Remove {both_empty.sum():,} emails with no content")
if len(potential_duplicates) > 100:
    print(f"  - Consider deduplication strategy for {len(potential_duplicates):,} potential duplicates")

print(f"\nUsable emails: {len(df_sample) - both_empty.sum():,}")

## 1.5 Workload State Simulation Strategy

Following instructor feedback to cluster email timestamps, let's analyze email volume patterns to identify busy vs. quiet periods.

In [ ]:
# Analyze email volume patterns for workload simulation
if 'datetime' in df_sample.columns:
    # Group by sender and hour to understand workload patterns
    df_sample['date_only'] = df_sample['datetime'].dt.date
    
    # Calculate emails per sender per day
    sender_daily_volume = df_sample.groupby(['from', 'date_only']).size().reset_index(name='daily_email_count')
    
    print("Sender Daily Email Volume Statistics:")
    print(sender_daily_volume['daily_email_count'].describe())
    
    # Visualize volume distribution
    plt.figure(figsize=(12, 6))
    plt.hist(sender_daily_volume['daily_email_count'], bins=50, edgecolor='black', alpha=0.7)
    plt.xlabel('Emails per Day (per sender)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Daily Email Volume per Sender')
    plt.axvline(sender_daily_volume['daily_email_count'].median(), 
                color='red', linestyle='--', label='Median')
    plt.axvline(sender_daily_volume['daily_email_count'].quantile(0.75), 
                color='orange', linestyle='--', label='Q3 (Busy threshold)')
    plt.legend()
    plt.show()
    
    # Define busy vs quiet thresholds
    busy_threshold = sender_daily_volume['daily_email_count'].quantile(0.75)
    quiet_threshold = sender_daily_volume['daily_email_count'].quantile(0.25)
    
    print(f"\nProposed Workload Thresholds:")
    print(f"  Quiet period: < {quiet_threshold:.0f} emails/day")
    print(f"  Normal period: {quiet_threshold:.0f} - {busy_threshold:.0f} emails/day")
    print(f"  Busy period: > {busy_threshold:.0f} emails/day")
    
    print("\nThis clustering approach will be used to synthesize workload states for annotation.")

## 1.6 Next Steps

Based on this exploration, we will:

1. **Clean and preprocess** the full dataset
2. **Select diverse sample** of emails for annotation (2,000-5,000)
3. **Create annotation pipeline** using Groq API with contextual scenarios
4. **Implement baseline models** (metadata-only, text-only, combined)
5. **Build context-aware models** incorporating temporal and workload features

---

**Key Insights from Exploration:**
- Dataset is rich with temporal patterns (peak hours, weekday patterns)
- Email lengths vary significantly (use for feature engineering)
- Urgency keywords present in ~X% of emails
- Clear workload patterns exist for busy/quiet period classification
- Data quality is generally good with minimal empty content

In [ ]:
# Save cleaned sample for next notebook
print("Saving cleaned sample...")
df_sample.to_csv('data_exploration_sample.csv', index=False)
print(f"Saved {len(df_sample):,} emails to 'data_exploration_sample.csv'")

print("\n" + "="*80)
print("Data Exploration Complete!")
print("="*80)
print("\nReady to proceed to Annotation Pipeline (Notebook 2)")